In [1]:
import os
import torch
import time
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A

class AugmentDataset(Dataset):
    def __init__(self, csv_path, image_dir, mask_dir, split='train', augment=True):
        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.augment = augment

    def randomHorizontalFlip(self, image, mask, u=0.5):
        if np.random.rand() < u:
            image = cv2.flip(image, 1)
            mask = cv2.flip(mask, 1)
        return image, mask

    def randomVerticalFlip(self, image, mask, u=0.5):
        if np.random.rand() < u:
            image = cv2.flip(image, 0)
            mask = cv2.flip(mask, 0)
        return image, mask

    def randomRotate90(self, image, mask, u=0.5):
        if np.random.rand() < u:
            image = np.rot90(image).copy()
            mask = np.rot90(mask).copy()
        return image, mask

    def randomHueSaturationValue(self, image, hue_shift_limit=(-30, 30),
                                  sat_shift_limit=(-5, 5),
                                  val_shift_limit=(-15, 15), u=0.5):
        if np.random.rand() < u:
            image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
            h, s, v = cv2.split(image)
            h = h.astype(np.float32)
            s = s.astype(np.float32)
            v = v.astype(np.float32)


            h += np.random.randint(hue_shift_limit[0], hue_shift_limit[1] + 1)
            s += np.random.uniform(sat_shift_limit[0], sat_shift_limit[1])
            v += np.random.uniform(val_shift_limit[0], val_shift_limit[1])

            h = np.clip(h, 0, 179).astype(np.uint8)
            s = np.clip(s, 0, 255).astype(np.uint8)
            v = np.clip(v, 0, 255).astype(np.uint8)

            image = cv2.merge((h, s, v))
            image = cv2.cvtColor(image, cv2.COLOR_HSV2RGB)
        return image

    def randomShiftScaleRotate(self, image, mask,
                               shift_limit=(-0.1, 0.1),
                               scale_limit=(-0.1, 0.1),
                               rotate_limit=(0, 0),
                               aspect_limit=(-0.1, 0.1),
                               borderMode=cv2.BORDER_CONSTANT, u=0.5):
        if np.random.rand() < u:
            height, width = image.shape[:2]
            angle = np.random.uniform(*rotate_limit)
            scale = np.random.uniform(1 + scale_limit[0], 1 + scale_limit[1])
            aspect = np.random.uniform(1 + aspect_limit[0], 1 + aspect_limit[1])
            sx = scale * aspect / (aspect ** 0.5)
            sy = scale / (aspect ** 0.5)
            dx = int(np.random.uniform(*shift_limit) * width)
            dy = int(np.random.uniform(*shift_limit) * height)

            cc = np.cos(np.radians(angle)) * sx
            ss = np.sin(np.radians(angle)) * sy
            rotation = np.array([[cc, -ss], [ss, cc]])

            box = np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32)
            box -= np.array([width / 2, height / 2], dtype=np.float32)
            box = np.dot(box, rotation.T) + np.array([width / 2 + dx, height / 2 + dy], dtype=np.float32)

            mat = cv2.getPerspectiveTransform(box.astype(np.float32), np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32))
            image = cv2.warpPerspective(image, mat, (width, height), flags=cv2.INTER_LINEAR, borderMode=borderMode, borderValue=(0, 0, 0))
            mask = cv2.warpPerspective(mask, mat, (width, height), flags=cv2.INTER_NEAREST, borderMode=borderMode, borderValue=(0,))

        return image, mask
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filename'])
        mask_path = os.path.join(self.mask_dir, row['maskname'])

        # Read image & mask
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        image = cv2.resize(image, (512, 512), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)

        # Augment
        if self.augment:
            image = self.randomHueSaturationValue(image)
            image, mask = self.randomShiftScaleRotate(image, mask)
            image, mask = self.randomHorizontalFlip(image, mask)
            image, mask = self.randomVerticalFlip(image, mask)
            image, mask = self.randomRotate90(image, mask)

        # Normalize image
        image = image.astype(np.float32) / 255.0
        image = image * 3.2 - 1.6
        image = np.transpose(image, (2, 0, 1))  # HWC → CHW

        # Normalize mask
        mask = mask.astype(np.float32) / 255.0
        mask = np.expand_dims(mask, axis=0)  # (1, H, W)
        mask = (mask > 0.5).astype(np.float32)

        return torch.tensor(image), torch.tensor(mask)


In [3]:
import torch.nn.functional as F

In [4]:
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

def get_optimizer(model, lr=5e-4, weight_decay=1e-4):
    return AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

def get_scheduler(optimizer, T_0=10, T_mult=2, eta_min=1e-6):
    return CosineAnnealingWarmRestarts(
    optimizer, 
    T_0=T_0, 
    T_mult=T_mult, 
    eta_min=eta_min
)

In [5]:
def bce_dice_loss(y_pred_logits, y_true, smooth=1e-5):
    """
    y_pred_logits: Raw model outputs (NO Sigmoid applied in the architecture)
    y_true: Ground truth binary masks
    """

    logits = y_pred_logits.logits
    logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

    # 1. Numerically stable BCE using Logits
    bce = F.binary_cross_entropy_with_logits(logits, y_true)

    # 2. Apply Sigmoid safely inside the loss for Dice calculation
    y_pred_probs = torch.sigmoid(logits)

    # 3. Flatten the ENTIRE batch into a single 1D vector for maximum GPU speed
    y_pred_flat = y_pred_probs.view(-1)
    y_true_flat = y_true.view(-1)

    # 4. Global Dice calculation
    intersection = (y_pred_flat * y_true_flat).sum()
    dice = (2. * intersection + smooth) / (y_pred_flat.sum() + y_true_flat.sum() + smooth)

    dice_loss = 1.0 - dice

    # Combined loss
    return bce + dice_loss

In [6]:
import torch
import os

def save_model(model, path, epoch, score):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    state = {
        'model_state_dict': model.state_dict(),
        'epoch': epoch,
        'score': score
    }
    torch.save(state, path)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
# === Cấu hình ===
batch_size = 4
num_epochs = 41
lr = 5e-5
log_dir = '/kaggle/working/logs'
model_path = '/kaggle/working/best_model.pth'
csv_path = '/kaggle/input/datasets/vuongtran11233/new-division/data_division.csv'
image_dir = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/images'
mask_dir = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/masks'

In [9]:
# === Dataloader ===
train_dataset = AugmentDataset(csv_path, image_dir, mask_dir, split='train', augment=True)
val_dataset = AugmentDataset(csv_path, image_dir, mask_dir, split='val', augment=False)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4,pin_memory=True,drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4,pin_memory=True,drop_last=True)

In [ ]:
# Previous-run best model
input_model_path = '/kaggle/input/models/vngtrnml17/segformer-b2-v1/pytorch/default/1/best_segformer.pth'

In [ ]:
from collections import OrderedDict
import torch
from collections import OrderedDict
from transformers import SegformerForSemanticSegmentation

# 1. Initialize the base architecture from Hugging Face
# This sets up the structure and handles the new class count (num_labels=1)
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-cityscapes-1024-1024",
    num_labels=1,
    ignore_mismatched_sizes=True
)

# 2. Load your Kaggle .pth checkpoint
input_model_path = '/kaggle/input/models/vngtrnml17/segformer-b2-v1/pytorch/default/1/best_segformer.pth'
checkpoint = torch.load(input_model_path, map_location=device)

# 3. Handle the dictionary check (depending on how you saved it)
# If you didn't save it with a 'model_state_dict' wrapper, just use checkpoint directly
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

# 4. Strip the 'module.' prefix from DataParallel
clean_state_dict = OrderedDict()
for key, value in state_dict.items():
    if key.startswith('module.'):
        clean_state_dict[key[7:]] = value
    else:
        clean_state_dict[key] = value

# 5. Load the weights into the model
# CRITICAL: strict=False allows it to ignore mismatched head weights smoothly
missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# Optional: Print to ensure everything loaded smoothly except the classification head
print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

model.to(device)

In [ ]:
'''from transformers import SegformerForSemanticSegmentation

# Initialize pre-trained SegFormer-B2
# Setting num_labels=1 drops the default 19-class Cityscapes head and replaces it with a 1-channel head
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-cityscapes-1024-1024",
    num_labels=1,
    ignore_mismatched_sizes=True
)'''

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/110M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-cityscapes-1024-1024
Key                           | Status   |                                                                                                  
------------------------------+----------+--------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([19, 768, 1, 1]) vs model:torch.Size([1, 768, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([19]) vs model:torch.Size([1])                      

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [11]:
# === Model, Loss, Optimizer ===
#model = DeepLabV3PlusResNet34(num_classes=1).to(device)
'''if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs for parallel training!")
    model = nn.DataParallel(model)'''
model.to(device)
optimizer = get_optimizer(model, lr)
scheduler = get_scheduler(optimizer)
EarlyStopPatience = 10

In [12]:
import gc

In [13]:
import csv
import os

# Create a custom log file in the output directory
log_file_path = "/kaggle/working/training_history.csv"

# Write headers if file doesn't exist
if not os.path.exists(log_file_path):
    with open(log_file_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss"])

# Inside your training loop at the end of every epoch:
def append_log(epoch, train_loss, val_loss):
    with open(log_file_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, train_loss, val_loss])

In [14]:
# === Training Loop ===
best_loss = 999.0
for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader, desc='Training'):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = bce_dice_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # === Validation ===
    model.eval()
    val_loss = 0
    all_preds = []
    all_masks = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc='Validation'):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = bce_dice_loss(outputs, masks)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    append_log(epoch + 1, train_loss, val_loss)

    # === Scheduler + EarlyStopping ===
    scheduler.step(val_loss)
    
    if val_loss < best_loss:
        best_loss = val_loss
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        save_model(model, model_path, epoch, best_loss)
        patience_counter = 0  # Reset counter since we improved
        print("Validation loss improved!")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{EarlyStopPatience}")

    # Trigger early stopping
    if patience_counter >= EarlyStopPatience:
        print("Early stopping triggered. Stopping training!")
        break

    del outputs, loss
    gc.collect()
    torch.cuda.empty_cache()

print("Training completed!")



Epoch [1/41]



Validation: 100%|██████████| 277/277 [00:58<00:00,  4.75it/s]


Train Loss: 0.7951 | Val Loss: 0.5526
Validation loss improved!

Epoch [2/41]


Validation: 100%|██████████| 277/277 [00:58<00:00,  4.73it/s]


Train Loss: 0.5180 | Val Loss: 0.4843
Validation loss improved!

Epoch [3/41]


...

Train Loss: 0.3415 | Val Loss: 0.3618
Validation loss improved!

Epoch [40/41]


Validation: 100%|██████████| 277/277 [00:58<00:00,  4.75it/s]


Train Loss: 0.3393 | Val Loss: 0.3646
No improvement. Patience: 1/10

Epoch [41/41]


Validation: 100%|██████████| 277/277 [00:58<00:00,  4.73it/s]


Train Loss: 0.3392 | Val Loss: 0.3635
No improvement. Patience: 2/10
Training completed!
